In [10]:
from pathlib import Path

LAST_NAME = "Fabia"
STUDENT_ID = "TUPM-26-1266"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)

print("Seed:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)

Seed: 6
ID Sum: 23
Name Length: 5


In [11]:
import csv

with open("data/user_behavior_dataset.csv", "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    print(reader.fieldnames)

['User ID', 'Device Model', 'Operating System', 'App Usage Time (min/day)', 'Screen On Time (hours/day)', 'Battery Drain (mAh/day)', 'Number of Apps Installed', 'Data Usage (MB/day)', 'Age', 'Gender', 'User Behavior Class']


In [12]:
import csv

with open("data/user_behavior_dataset.csv", "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for header in reader.fieldnames:
        print(header)

User ID
Device Model
Operating System
App Usage Time (min/day)
Screen On Time (hours/day)
Battery Drain (mAh/day)
Number of Apps Installed
Data Usage (MB/day)
Age
Gender
User Behavior Class


In [13]:
from pathlib import Path
import csv
import json
import shutil

# Student Values
SEED_DIGIT = 6

# Generated Values
TARGET_OS = "Android"
TARGET_CLASS = (SEED_DIGIT % 5) + 1

# Folders
data_dir = Path("data")
output_dir = Path("output")
backup_dir = Path("backup")

output_dir.mkdir(exist_ok=True)
backup_dir.mkdir(exist_ok=True)

# Dataset
dataset_path = data_dir / "user_behavior_dataset.csv"

selected_users = []

total_records = 0
matching_os = 0

battery_total = 0
data_usage_total = 0

with open(dataset_path, "r", encoding="utf-8") as file:

    reader = csv.DictReader(file)

    for row in reader:

        total_records += 1

        if row["Operating System"] == TARGET_OS:

            matching_os += 1

            if int(row["User Behavior Class"]) == TARGET_CLASS:

                selected_users.append(row)

                battery_total += float(
                    row["Battery Drain (mAh/day)"]
                )

                data_usage_total += float(
                    row["Data Usage (MB/day)"]
                )

record_count = len(selected_users)

if record_count > 0:
    avg_battery = battery_total / record_count
    avg_data_usage = data_usage_total / record_count
else:
    avg_battery = 0
    avg_data_usage = 0

# Export CSV
csv_file = output_dir / "mobile_profile.csv"

fields = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain",
    "Data Usage"
]

with open(csv_file, "w", newline="", encoding="utf-8") as file:

    writer = csv.DictWriter(file, fieldnames=fields)

    writer.writeheader()

    for row in selected_users:

        writer.writerow({
            "User ID": row["User ID"],
            "Device Model": row["Device Model"],
            "Operating System": row["Operating System"],
            "Battery Drain": row["Battery Drain (mAh/day)"],
            "Data Usage": row["Data Usage (MB/day)"]
        })

# Create Summary
summary = {
    "Operating System": TARGET_OS,
    "Behavior Class": TARGET_CLASS,
    "Total Records": total_records,
    "Records Matching OS": matching_os,
    "Records in Target Class": record_count,
    "Average Battery Drain": round(avg_battery, 2),
    "Average Data Usage": round(avg_data_usage, 2),
    "Exported Rows": record_count
}

json_file = output_dir / "mobile_summary.json"

with open(json_file, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

# Backup CSV
backup_file = backup_dir / csv_file.name

shutil.copy2(csv_file, backup_file)

# Results
print("Operating System:", TARGET_OS)
print("Behavior Class:", TARGET_CLASS)
print("Total Records:", total_records)
print("Records Matching OS:", matching_os)
print("Records in Target Class:", record_count)
print("Average Battery Drain (mAh/day):", round(avg_battery, 2))
print("Average Data Usage (MB/day):", round(avg_data_usage, 2))
print("Count Verification:", record_count)
print("Exported CSV:", csv_file)
print("JSON Summary:", json_file)
print("Backup Exists:", backup_file.exists())
print("Backup Size:", backup_file.stat().st_size, "bytes")

Operating System: Android
Behavior Class: 2
Total Records: 700
Records Matching OS: 554
Records in Target Class: 114
Average Battery Drain (mAh/day): 887.11
Average Data Usage (MB/day): 451.44
Count Verification: 114
Exported CSV: output\mobile_profile.csv
JSON Summary: output\mobile_summary.json
Backup Exists: True
Backup Size: 4098 bytes
